In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path


In [ ]:
# 0) 필요한 패키지(최초 1회만)
# pip install geopandas shapely pyproj matplotlib contextily

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path

# 1) 파일 경로 지정 (여기만 바꾸세요)
shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')  # 예시(윈도우)
# shp_path = Path("/Users/you/Downloads/읍면동_경계.shp")  # 예시(mac)

gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 한글 인코딩 주의
dunsan = gdf[gdf['EMD_NM'] == '둔산동']

dunsan_4326 = dunsan.to_crs(epsg=4326)
dunsan_4326


In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 50)


df = pd.read_excel('./data/둔산동 내 기존 폐의약품 수거함_총인구수 및 65세 이상.xlsx')
df.head(30)

In [ ]:
import folium

b = dunsan_4326.total_bounds  # [minx, miny, maxx, maxy]
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m)
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
m


In [ ]:
gdf_origin = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["경도"], df["위도"]),
    crs="EPSG:4326"
)
gdf_origin.head(2)

for _, row in gdf_origin.iterrows():
    lat = row["위도"]
    lon = row["경도"]
    name = row["수거장소명"]

    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        tooltip=name  # 마우스 올리면 상호명 표시
    ).add_to(m)

m